## Random Forest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn modules
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# XGBoost
import xgboost as xgb

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load dataset
wine_data = load_wine()
X = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
y = wine_data.target

print(f"Dataset shape: {X.shape}")
print(f"Classes: {wine_data.target_names}")
print(f"\nFeature names:")
for i, name in enumerate(wine_data.feature_names):
    print(f"  {i+1}. {name}")

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

In [ ]:
# Train a simple Decision Tree
dt_simple = DecisionTreeClassifier(random_state=42)
dt_simple.fit(X_train, y_train)
y_pred_dt = dt_simple.predict(X_test)
dt_acc = accuracy_score(y_test, y_pred_dt)

# Train Random Forest with default parameters
rf_simple = RandomForestClassifier(n_estimators=100, random_state=42)
rf_simple.fit(X_train, y_train)
y_pred_rf = rf_simple.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)

print(f"Decision Tree Accuracy: {dt_acc:.4f}")
print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"Improvement: {(rf_acc - dt_acc):.4f} ({((rf_acc - dt_acc) / dt_acc * 100):.2f}%)")

# Compare training vs test accuracy to see overfitting
dt_train_acc = accuracy_score(y_train, dt_simple.predict(X_train))
rf_train_acc = accuracy_score(y_train, rf_simple.predict(X_train))

print(f"\nDecision Tree - Train: {dt_train_acc:.4f}, Test: {dt_acc:.4f}, Overfitting Gap: {(dt_train_acc - dt_acc):.4f}")
print(f"Random Forest - Train: {rf_train_acc:.4f}, Test: {rf_acc:.4f}, Overfitting Gap: {(rf_train_acc - rf_acc):.4f}")

# Visualize comparison
comparison_df = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Train Accuracy': [dt_train_acc, rf_train_acc],
    'Test Accuracy': [dt_acc, rf_acc]
})

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Train Accuracy'], width, label='Train Accuracy', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison_df['Test Accuracy'], width, label='Test Accuracy', alpha=0.8)

ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree vs Random Forest: Overfitting Comparison')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'])
ax.legend()
ax.set_ylim([0.8, 1.05])

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Boosting

In [ ]:
# Initialize AdaBoost
ada_classifier = AdaBoostClassifier(n_estimators=100, random_state=42)
ada_classifier.fit(X_train, y_train)
y_pred_ada = ada_classifier.predict(X_test)
ada_acc = accuracy_score(y_test, y_pred_ada)

print(f"AdaBoost Accuracy: {ada_acc:.4f}")

# Initialize Gradient Boosting
gb_classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_classifier.fit(X_train, y_train)
y_pred_gb = gb_classifier.predict(X_test)
gb_acc = accuracy_score(y_test, y_pred_gb)

print(f"Gradient Boosting Accuracy: {gb_acc:.4f}")

# Initialize XGBoost
xgb_classifier = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb_classifier.fit(X_train, y_train)
y_pred_xgb = xgb_classifier.predict(X_test)
xgb_acc = accuracy_score(y_test, y_pred_xgb)

print(f"XGBoost Accuracy: {xgb_acc:.4f}")

In [ ]:
# Comprehensive model comparison
all_results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest (Default)', 'Random Forest (Tuned)', 'AdaBoost', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [dt_acc, rf_acc, best_rf_acc, ada_acc, gb_acc, xgb_acc]
})

# Sort by accuracy
all_results = all_results.sort_values('Accuracy', ascending=False)

print("\n" + "="*60)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*60)
print(all_results.to_string(index=False))

# Visualize comparison
plt.figure(figsize=(12, 6))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(all_results))]
bars = plt.barh(all_results['Model'], all_results['Accuracy'], color=colors)

plt.xlabel('Accuracy', fontsize=12)
plt.title('Model Accuracy Comparison on Wine Dataset', fontsize=14, fontweight='bold')
plt.xlim([0.8, 1.05])

# Add value labels
for i, (model, acc) in enumerate(zip(all_results['Model'], all_results['Accuracy'])):
    plt.text(acc + 0.005, i, f'{acc:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Comparison

In [ ]:
# Get feature importances from multiple models
feature_names = wine_data.feature_names

# Extract importances
dt_importances = dt_simple.feature_importances_
rf_importances = best_rf.feature_importances_
gb_importances = gb_classifier.feature_importances_
xgb_importances = xgb_classifier.feature_importances_

# Create comparison DataFrame
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Decision Tree': dt_importances,
    'Random Forest': rf_importances,
    'Gradient Boosting': gb_importances,
    'XGBoost': xgb_importances
})

# Sort by Random Forest importance
feat_imp_df = feat_imp_df.sort_values('Random Forest', ascending=False)

print("\nFeature Importance Comparison:")
print(feat_imp_df.to_string(index=False))

# Visualize feature importance comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models = ['Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost']
axes_flat = axes.flatten()

for idx, (ax, model) in enumerate(zip(axes_flat, models)):
    top_features = feat_imp_df.nlargest(8, model)
    ax.barh(range(len(top_features)), top_features[model].values, color='steelblue')
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features['Feature'].values)
    ax.set_xlabel('Importance')
    ax.set_title(f'{model} - Top 8 Features')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()